In [1]:
from google.cloud import bigquery

PROJECT = "gridzero-489711"
DATASET = "merged_set"
TABLE = "full_feature_engineered_data_test"

query = f"""
    SELECT *
    FROM {PROJECT}.{DATASET}.{TABLE}
"""

client = bigquery.Client('gridzero-489711')
query_job = client.query(query)
result = query_job.result()
df = result.to_dataframe()

In [2]:
df.head()

,datetime,temperature_2m_c,wind_speed_100m_ms,wind_gusts_10m_ms,cloud_cover_pct,shortwave_radiation_wm2,direct_radiation_wm2,diffuse_radiation_wm2,pressure_msl_hpa,precipitation_mm,...,carbon_intensity_gco2_kwh,hour_sin,hour_cos,dow_sin,dow_cos,doy_sin,doy_cos,carbon_lag_48,carbon_lag_336,carbon_lag_17520
0,2017-09-12 00:00:00+00:00,11.6,31.0,28.1,4.0,0.0,0.0,0.0,1001.2,0.0,...,142.0,0.000000,1.000000,0.781831,0.62349,-0.948362,-0.317191,NaN,NaN,NaN
1,2017-09-12 00:30:00+00:00,11.6,31.0,28.1,4.0,0.0,0.0,0.0,1001.2,0.0,...,140.0,0.000000,1.000000,0.781831,0.62349,-0.948362,-0.317191,NaN,NaN,NaN
2,2017-09-12 01:00:00+00:00,11.2,30.3,27.0,5.0,0.0,0.0,0.0,1001.9,0.0,...,139.0,0.258819,0.965926,0.781831,0.62349,-0.948362,-0.317191,NaN,NaN,NaN
3,2017-09-12 01:30:00+00:00,11.2,30.3,27.0,5.0,0.0,0.0,0.0,1001.9,0.0,...,137.0,0.258819,0.965926,0.781831,0.62349,-0.948362,-0.317191,NaN,NaN,NaN
4,2017-09-12 02:00:00+00:00,10.9,29.6,25.2,7.0,0.0,0.0,0.0,1002.4,0.0,...,132.0,0.500000,0.866025,0.781831,0.62349,-0.948362,-0.317191,NaN,NaN,NaN


In [3]:
df.columns

Index(['datetime', 'temperature_2m_c', 'wind_speed_100m_ms',
       'wind_gusts_10m_ms', 'cloud_cover_pct', 'shortwave_radiation_wm2',
       'direct_radiation_wm2', 'diffuse_radiation_wm2', 'pressure_msl_hpa',
       'precipitation_mm', 'biomass', 'fossil_gas', 'fossil_hard_coal',
       'hydro_pumped_storage', 'hydro_run_of_river_and_poundage', 'nuclear',
       'other', 'solar', 'wind_offshore', 'wind_onshore', 'totaloutput_mw',
       'carbon_intensity_gco2_kwh', 'hour_sin', 'hour_cos', 'dow_sin',
       'dow_cos', 'doy_sin', 'doy_cos', 'carbon_lag_48', 'carbon_lag_336',
       'carbon_lag_17520'],
      dtype='object')

In [5]:
feature_cols = [
    # weather
    'temperature_2m_c',
    'wind_speed_100m_ms',
    'wind_gusts_10m_ms',
    'cloud_cover_pct',
    'shortwave_radiation_wm2',
    'direct_radiation_wm2',
    'diffuse_radiation_wm2',
    'pressure_msl_hpa',
    'precipitation_mm',

    # time
    'hour_sin','hour_cos',
    'dow_sin','dow_cos',
    'doy_sin','doy_cos',
    
    # past generation (important)
    'biomass',
    'fossil_gas',
    'fossil_hard_coal',
    'hydro_pumped_storage',
    'hydro_run_of_river_and_poundage',
    'nuclear',
    'other',
    'solar',
    'wind_offshore',
    'wind_onshore'
]

In [ ]:
# Index(['datetime', 'temperature_2m', 'wind_gusts_10m', 'cloud_cover',
#        'direct_radiation', 'diffuse_radiation', 'shortwave_radiation',
#        'pressure_msl', 'precipitation', 'wind_speed_100m', 'biomass',
#        'fossil_gas', 'fossil_hard_coal', 'fossil_oil', 'hydro_pumped_storage',
#        'hydro_run_of_river_and_poundage', 'nuclear', 'other', 'solar',
#        'wind_offshore', 'wind_onshore', 'total_output_mw', 'hour_sin',
#        'hour_cos', 'dow_sin', 'dow_cos', 'doy_sin', 'doy_cos'],
#       dtype='object')

In [ ]:
# datetime, totalmw, order them

In [6]:
target_cols = [
    'biomass',
    'fossil_gas',
    'fossil_hard_coal',
    'hydro_pumped_storage',
    'hydro_run_of_river_and_poundage',
    'nuclear',
    'other',
    'solar',
    'wind_offshore',
    'wind_onshore'
]

In [7]:
from sklearn.preprocessing import MinMaxScaler

X_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

X = X_scaler.fit_transform(df[feature_cols])
y = y_scaler.fit_transform(df[target_cols])

In [8]:
import numpy as np

lookback = 336

def create_sequences(X, y, lookback):
    
    Xs, ys = [], []
    
    for i in range(len(X) - lookback):
        Xs.append(X[i:i+lookback])
        ys.append(y[i+lookback])
        
    return np.array(Xs), np.array(ys)

X_seq, y_seq = create_sequences(X, y, lookback)

In [9]:
X_seq.shape, y_seq.shape

((148655, 336, 25), (148655, 10))

In [10]:
train_size = int(len(X_seq) * 0.7)
val_size = int(len(X_seq) * 0.15)

X_train = X_seq[:train_size]
y_train = y_seq[:train_size]

X_val = X_seq[train_size:train_size+val_size]
y_val = y_seq[train_size:train_size+val_size]

X_test = X_seq[train_size+val_size:]
y_test = y_seq[train_size+val_size:]

## Building the LSTM model

In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential()

model.add(LSTM(128, return_sequences=True,
               input_shape=(lookback, len(feature_cols))))

model.add(Dropout(0.2))

model.add(LSTM(64))

model.add(Dense(64, activation="relu"))

model.add(Dense(len(target_cols)))

I0000 00:00:1773662992.340417   33469 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773662992.341325   33469 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1773662992.381645   33469 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773662993.677088   33469 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

In [12]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss="mse",
    metrics=["mae"]
)

In [13]:
# model.compile(
#     optimizer=Adam(learning_rate=0.001),
#     loss="mse"
# )

In [14]:
# history = model.fit(
#     X_train,
#     y_train,
#     validation_data=(X_val, y_val),
#     epochs=40,
#     batch_size=64
# )

In [15]:
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint(
    filepath="checkpoints/best_model.keras",
    monitor="val_loss",
    save_best_only=True,
    mode="min",
    verbose=1
)

In [16]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

In [36]:
X_train

array([[[0.45435685, 0.31062124, 0.24398931, ..., 0.        ,
         0.30761008, 0.39085052],
        [0.45435685, 0.31062124, 0.24398931, ..., 0.        ,
         0.30516107, 0.392938  ],
        [0.44605809, 0.30360721, 0.23419412, ..., 0.        ,
         0.30273468, 0.40312713],
        ...,
        [0.43775934, 0.13026052, 0.09973286, ..., 0.        ,
         0.06932427, 0.05066228],
        [0.43360996, 0.15330661, 0.08014248, ..., 0.        ,
         0.07634372, 0.04885181],
        [0.43360996, 0.15330661, 0.08014248, ..., 0.        ,
         0.08517617, 0.04861036]],

       [[0.45435685, 0.31062124, 0.24398931, ..., 0.        ,
         0.30516107, 0.392938  ],
        [0.44605809, 0.30360721, 0.23419412, ..., 0.        ,
         0.30273468, 0.40312713],
        [0.44605809, 0.30360721, 0.23419412, ..., 0.        ,
         0.30302942, 0.42030616],
        ...,
        [0.43360996, 0.15330661, 0.08014248, ..., 0.        ,
         0.07634372, 0.04885181],
        [0.4

In [ ]:
model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=64,
    callbacks=[checkpoint, early_stop]
)

Epoch 1/50
1626/1626 ━━━━━━━━━━━━━━━━━━━━ 0s 302ms/step - loss: 0.0099 - mae: 0.0631
Epoch 1: val_loss improved from None to 0.00369, saving model to checkpoints/best_model.keras
1626/1626 ━━━━━━━━━━━━━━━━━━━━ 532s 326ms/step - loss: 0.0049 - mae: 0.0448 - val_loss: 0.0037 - val_mae: 0.0373
Epoch 2/50
1626/1626 ━━━━━━━━━━━━━━━━━━━━ 0s 299ms/step - loss: 0.0023 - mae: 0.0308
Epoch 2: val_loss improved from 0.00369 to 0.00278, saving model to checkpoints/best_model.keras
1626/1626 ━━━━━━━━━━━━━━━━━━━━ 524s 322ms/step - loss: 0.0022 - mae: 0.0294 - val_loss: 0.0028 - val_mae: 0.0315
Epoch 3/50
1626/1626 ━━━━━━━━━━━━━━━━━━━━ 0s 298ms/step - loss: 0.0018 - mae: 0.0263
Epoch 3: val_loss improved from 0.00278 to 0.00248, saving model to checkpoints/best_model.keras
1626/1626 ━━━━━━━━━━━━━━━━━━━━ 522s 321ms/step - loss: 0.0018 - mae: 0.0256 - val_loss: 0.0025 - val_mae: 0.0284
Epoch 4/50
1626/1626 ━━━━━━━━━━━━━━━━━━━━ 0s 299ms/step - loss: 0.0016 - mae: 0.0240
Epoch 4: val_loss did not improve

In [23]:
model.save("gs://grid_zero_bucket/lstm_model1.keras")

In [24]:
# model.save("lstm_models/model1.keras")

In [25]:
# model = load_model("lstm_models/model2")

# loading model

In [ ]:
# blob.download_to_filename("/tmp/xgb_model.bin")
# model.load_model("/tmp/xgb_model.bin")